## Análisis y limpieza de ccnorte

El scraper (`ccnorte_scraper_completo.ipynb`) ya genera `ccnorte_participantes_resumen.csv` con todo lo necesario junto — `nombre_carrera`, `fecha`, `categoria`, recuentos por género y `categoria_genero` — así que aquí ya no hace falta ningún join. Solo queda: cargar, diagnosticar, renombrar, y clasificar `categoria` en disciplina (`tipo_modalidad`) y público por edad (`publico`), igual que hicimos con buscametas/carreirasgalegas.

In [1]:
from pathlib import Path
import pandas as pd

CSV_PATH = Path("../../data/raw/ccnorte/DF_CCNORTE_SUCIO.csv")
df = pd.read_csv(CSV_PATH)
df["fecha"] = pd.to_datetime(df["fecha"], errors="coerce")  # ya viene en ISO (YYYY-MM-DD), el scraper la parsea

print("Filas x columnas:", df.shape)
print()
print(df.dtypes)
df.head()

Filas x columnas: (6280, 8)

nombre_carrera              object
fecha               datetime64[ns]
categoria                   object
n_masculino                  int64
n_femenino                   int64
n_desconocido                int64
n_total                      int64
categoria_genero            object
dtype: object


,nombre_carrera,fecha,categoria,n_masculino,n_femenino,n_desconocido,n_total,categoria_genero
0,"""200 CRESTAS"" DESAFÍO COUREL 2016",2016-03-19,92 KM (Desnivel positivo: 3.702 metros - Desni...,201,2,0,203,mixto
1,"""I TRAIL 175º ANIVERSARIO GUARDIA CIVIL"" A BEN...",2019-05-12,TRAIL,159,33,0,192,mixto
2,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA A (2006-2008),8,4,0,12,mixto
3,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA B (2003-2005),4,10,0,14,mixto
4,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA C (2000-2002),7,5,0,12,mixto


In [2]:
# Diagnóstico antes de limpiar: nulos, duplicados y consistencia de conteos
print("Valores nulos por columna:")
print(df.isna().sum())
print()

print("Filas completamente duplicadas:", df.duplicated().sum())
print("Filas duplicadas por (nombre_carrera, categoria):",
      df.duplicated(subset=["nombre_carrera", "categoria"]).sum())
print()

print("Eventos (nombre_carrera) distintos:", df["nombre_carrera"].nunique())
print()

print("n_masculino + n_femenino + n_desconocido vs n_total, deberían coincidir:")
suma = df["n_masculino"] + df["n_femenino"] + df["n_desconocido"]
print((suma != df["n_total"]).sum(), "filas no coinciden")
print()

print("Valores de 'categoria' más frecuentes:")
print(df["categoria"].value_counts().head(40))

Valores nulos por columna:
nombre_carrera      0
fecha               0
categoria           0
n_masculino         0
n_femenino          0
n_desconocido       0
n_total             0
categoria_genero    0
dtype: int64

Filas completamente duplicadas: 0
Filas duplicadas por (nombre_carrera, categoria): 3

Eventos (nombre_carrera) distintos: 2479

n_masculino + n_femenino + n_desconocido vs n_total, deberían coincidir:
0 filas no coinciden

Valores de 'categoria' más frecuentes:
categoria
ABSOLUTA                               62
POPULAR                                51
FEMENINA                               41
Carrera                                36
MASCULINA                              35
Femenina                               34
ELITE                                  34
PROBA ABSOLUTA. DISTANCIA: 10000 M     32
TRAIL                                  25
MEDIO MARATON                          25
Masculina                              24
INDIVIDUAL                             22
ALEVIN

In [3]:
# Limpieza: nos quedamos con las columnas que interesan, renombradas.
# finisher_desconocido se conserva como columna propia (aquí puede pesar
# más que en los otros proyectos). circuito_nombre ya no existe (el
# scraper la descarta); categoria_genero viene ya calculada por el
# scraper. finisher_d/finisher_h: mismo nombre que en el resto de fuentes
# (antes ok_d/ok_h).
curses_limpio = df[
    ["nombre_carrera", "fecha", "categoria", "categoria_genero", "n_femenino", "n_masculino", "n_desconocido"]
].rename(columns={
    "categoria": "modalidad",
    "n_femenino": "finisher_d",
    "n_masculino": "finisher_h",
    "n_desconocido": "finisher_desconocido",
})

print(curses_limpio.shape)
curses_limpio.head()

(6280, 7)


,nombre_carrera,fecha,modalidad,categoria_genero,finisher_d,finisher_h,finisher_desconocido
0,"""200 CRESTAS"" DESAFÍO COUREL 2016",2016-03-19,92 KM (Desnivel positivo: 3.702 metros - Desni...,mixto,2,201,0
1,"""I TRAIL 175º ANIVERSARIO GUARDIA CIVIL"" A BEN...",2019-05-12,TRAIL,mixto,33,159,0
2,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA A (2006-2008),mixto,4,8,0
3,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA B (2003-2005),mixto,10,4,0
4,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA C (2000-2002),mixto,5,7,0


In [4]:
import re

# "modalidad" aquí trae ruido: desnivel entre paréntesis ("92 KM (Desnivel
# positivo: 3.702 metros...)") y etiquetas de estado como "(STARTED)". Lo
# quitamos antes de sacar la distancia y limpiar el texto.
curses_limpio["modalidad"] = curses_limpio["modalidad"].str.replace(
    r"\([^)]*\)", "", regex=True
).str.strip()

# La distancia puede venir en K/KM ("10K", "92 KM"), explícita en metros con
# la palabra "distancia" ("DISTANCIA: 10000 M") o en metros sueltos sin esa
# palabra ("1200m", "PRUEBA 800m", "2500 MTS" — muy habitual en las
# travesías a nado).
distancia_km = curses_limpio["modalidad"].str.extract(r"(\d+(?:[.,]\d+)?)\s*[kK]")[0]
distancia_m_explicita = curses_limpio["modalidad"].str.extract(
    r"distancia:?\s*(\d+(?:[.,]\d+)?)\s*m\b", flags=re.IGNORECASE
)[0]
distancia_m_suelta = curses_limpio["modalidad"].str.extract(
    r"(\d+(?:[.,]\d+)?)\s*m(?:ts|etros)?\b", flags=re.IGNORECASE
)[0]

curses_limpio["distancia"] = distancia_km.str.replace(",", ".", regex=False).astype(float)

_faltan = curses_limpio["distancia"].isna()
curses_limpio.loc[_faltan, "distancia"] = (
    distancia_m_explicita[_faltan].str.replace(",", ".", regex=False).astype(float) / 1000
)

_faltan = curses_limpio["distancia"].isna()
curses_limpio.loc[_faltan, "distancia"] = (
    distancia_m_suelta[_faltan].str.replace(",", ".", regex=False).astype(float) / 1000
)

# "MM"/"MEDIO MARATÓN"/"MEDIA MARATÓN" y "MARATÓN" a secas tienen una
# distancia oficial conocida (21.097 km / 42.195 km) aunque no la pongan
# explícita ("MEDIO MARATON TRICICLO, HANDBIKE Y JOELETTE") — la usamos si
# seguimos sin distancia. Se comprueba primero "medio/media" para no
# confundirlo con el maratón completo (que también contiene "marat").
_falta_aun = curses_limpio["distancia"].isna()
_es_medio_maraton = curses_limpio["modalidad"].str.contains(
    r"medio\s*marat|media\s*marat|\bmm\b", case=False, regex=True, na=False
)
_es_maraton = curses_limpio["modalidad"].str.contains(r"marat", case=False, regex=True, na=False)

curses_limpio.loc[_falta_aun & _es_medio_maraton, "distancia"] = 21.097
curses_limpio.loc[_falta_aun & ~_es_medio_maraton & _es_maraton, "distancia"] = 42.195

# Igual que con tipo_modalidad: si "modalidad" no menciona el maratón pero
# el NOMBRE de la carrera sí ("XXV MEDIO MARATÓN GRAN BAHÍA..."), usamos
# ese respaldo antes de rendirnos.
_falta_aun = curses_limpio["distancia"].isna()
_nombre_es_medio_maraton = curses_limpio["nombre_carrera"].str.contains(
    r"medio\s*marat|media\s*marat|\bmm\b", case=False, regex=True, na=False
)
_nombre_es_maraton = curses_limpio["nombre_carrera"].str.contains(r"marat", case=False, regex=True, na=False)

curses_limpio.loc[_falta_aun & _nombre_es_medio_maraton, "distancia"] = 21.097
_falta_aun = curses_limpio["distancia"].isna()
curses_limpio.loc[_falta_aun & ~_nombre_es_medio_maraton & _nombre_es_maraton, "distancia"] = 42.195

curses_limpio["modalidad"] = (
    curses_limpio["modalidad"]
    .str.replace(r"\d+(?:[.,]\d+)?\s*[kK][mM]?\b", "", regex=True)
    .str.replace(r"distancia:?\s*\d+(?:[.,]\d+)?\s*m\b", "", regex=True, flags=re.IGNORECASE)
    .str.replace(r"\d+(?:[.,]\d+)?\s*m(?:ts|etros)?\b", "", regex=True, flags=re.IGNORECASE)
    .str.strip(" ,.-")
    # después de quitar el número, a veces queda una preposición colgando
    # ("RECORRIDO DE" -> "RECORRIDO", "COMPETICION DE" -> "COMPETICION")
    .str.replace(r"\s+(de|del|con)\s*$", "", regex=True, flags=re.IGNORECASE)
    .str.replace(r"\s{2,}", " ", regex=True)
)

print("Filas con distancia detectada:", curses_limpio["distancia"].notna().sum(),
      "de", len(curses_limpio))
print()
print("Ejemplos de modalidad SIN distancia detectada (revisa si falta algún patrón):")
print(curses_limpio.loc[curses_limpio["distancia"].isna(), "modalidad"].value_counts().head(30))
curses_limpio.head()

Filas con distancia detectada: 2131 de 6280

Ejemplos de modalidad SIN distancia detectada (revisa si falta algún patrón):
modalidad
ABSOLUTA                       328
ALEVIN                         102
ADULTOS                         73
BENXAMIN                        70
INFANTIL                        65
POPULAR                         61
INFANTIL E CADETE               53
TRAIL                           49
CADETE                          47
ELITE                           44
SUB12                           43
FEMENINA                        41
PROBA ABSOLUTA                  40
SUB10                           38
BENJAMIN                        37
CARREIRA ABSOLUTA               36
MASCULINA                       35
Femenina                        34
Carrera                         34
PREBENXAMIN                     32
SUB 12                          28
SUB14                           28
INDIVIDUAL                      26
SUB 10                          25
INFANTIL , CADETE E XUVENIL

,nombre_carrera,fecha,modalidad,categoria_genero,finisher_d,finisher_h,finisher_desconocido,distancia
0,"""200 CRESTAS"" DESAFÍO COUREL 2016",2016-03-19,,mixto,2,201,0,92.0
1,"""I TRAIL 175º ANIVERSARIO GUARDIA CIVIL"" A BEN...",2019-05-12,TRAIL,mixto,33,159,0,NaN
2,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA A,mixto,4,8,0,NaN
3,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA B,mixto,10,4,0,NaN
4,#MarzoSaudeSolidario 2015,2015-05-10,CARRERA CATEGORIA C,mixto,5,7,0,NaN


In [ ]:
# Clasificamos modalidad en disciplina (tipo_modalidad) por palabras clave,
# con respaldo en nombre_carrera si modalidad no dice nada.
import re

_CATEGORIAS = {
    "trail running": r"trail|trekking|vertical|\btra\b|\bkv\b|\butpd\b|joelette",
    "Ciclismo y btt": r"btt|ciclis|bici|mtb|gravel|ciclotur|e-?bike|handbike|triciclo|\bbike\b|gran ?fondo",
    # d[uú]atl/tr[ií]atl/[aá]cuatl: en gallego suelen llevar tilde en la
    # primera vocal ("Dúatlon", "Tríatlon", "Ácuatlon") — sin esto no encajaban.
    "Multidisciplina": r"d[uú]atl|tr[ií]atl|multidep|multidisci|aquatl|[aá]cuatl|combinada",
    "road running": r"carrera|carreira|running|popular|asfalto|ruta|marat|cross|cros|absoluta|10k|5k|21k|half|corredor|corre|lasterketa|campo a trav[eé]s|milla|[eé]lite|silvestre|\d+\s?km?\b|cadeira de rodas|silla de ruedas",
    "marcha": r"martxa|marcha|\bmar\b|andarin|andaina",
}

def _clasificar_texto(texto):
    for categoria, patron in _CATEGORIAS.items():
        if re.search(patron, texto):
            return categoria
    return None

def _clasificar(row):
    modalidad = row["modalidad"]
    texto_modalidad = "" if pd.isna(modalidad) else modalidad.lower().strip()
    categoria = _clasificar_texto(texto_modalidad) if texto_modalidad else None
    if categoria:
        return categoria

    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre.lower()
    categoria = _clasificar_texto(texto_nombre)
    if categoria:
        return categoria

    if texto_modalidad == "":
        return "road running"
    return "Otros"

curses_limpio["tipo_modalidad"] = curses_limpio.apply(_clasificar, axis=1)

print(curses_limpio["tipo_modalidad"].value_counts())
print()
print("Valores de 'modalidad' que han caído en 'Otros':")
print(curses_limpio.loc[curses_limpio["tipo_modalidad"] == "Otros", "modalidad"].value_counts())
print()
print("Nombres de carrera de las filas que siguen en 'Otros':")
print(curses_limpio.loc[curses_limpio["tipo_modalidad"] == "Otros", "nombre_carrera"].value_counts().head(30))

In [6]:
# Filas donde modalidad es SOLO "femenina"/"masculina" (sin nada más) y
# aun así siguen en Otros — es decir, el respaldo en nombre_carrera
# TAMPOCO ha encontrado ninguna pista. Miramos esos nombres de carrera
# tal cual, para ver si hay algún patrón que se nos escapa.
_mask_genero_puro = (
    curses_limpio["tipo_modalidad"].eq("Otros")
    & curses_limpio["modalidad"].str.lower().isin(["femenina", "masculina"])
)
print(f"{_mask_genero_puro.sum()} filas con modalidad femenina/masculina pura y sin pista en nombre_carrera")
print()
print(curses_limpio.loc[_mask_genero_puro, "nombre_carrera"].value_counts())

7 filas con modalidad femenina/masculina pura y sin pista en nombre_carrera

nombre_carrera
II Vertiatlón Solidario Concello de Baiona                                                             2
IV Vertiatlon Solidario Baiona                                                                         2
V Vertiatlon Solidario Baiona                                                                          2
ETAPA FINAL CIRCUITO GALEGO ARZUA ULLOA AUGAS ABERTAS-CAMPIONATO GALEGO CONTRA O RELOXO POR EQUIPOS    1
Name: count, dtype: int64


In [7]:
# Clasificamos el público (edad/audiencia), también con respaldo en
# nombre_carrera. "FEMENINA"/"MASCULINA" son splits por género (ya tenemos
# ok_d/ok_h reales), no dicen nada de edad, así que no cuentan aquí.
_OTROS_PATRON = (
    r"discapacidad|invidente|handbike|cadeira de rodas|silla de ruedas|"
    r"inscrici[oó]n|participaci[oó]n|promoci[oó]n|simulaci[oó]n|"
    r"material selecci[oó]n|^pago$|prueba de camiseta|asdfsadf|dsfasdfasd|"
    r"^\d+\s?k(m)?\b|^\d+\s?(fem|masc)\b"
)

_PUBLICOS = {
    "Infantil": (
        r"infant|benjam|benxam|alev|prebenjam|prebenxam|chupet|peque|pitufo|"
        r"biber[oó]n|familiar|menores|escolar|años|"
        r"sub\s?\d+\b|"
        r"cadete|juvenil|xuvenil|junior|cadet\b|promesa|preuniversitari"
    ),
    "Mayores/Veteranos": r"mayores|veteran|master|m[aá]ster",
    "Elite": r"\belite\b|profesional",
    # "EQUIPOS", "EQUIPOS FEMENINOS/MASCULINOS", "POR EQUIPOS" — categoría
    # de participación en grupo, no de edad, pero se distingue bien de las
    # demás y hay volumen suficiente para separarla de Absoluta/General.
    "Equipos": r"equipos?\b",
    "Absoluta/General": r"absoluta|\babs\b|general|popular|senior|adultos?|\bopen\b",
}

def _clasificar_publico_texto(texto):
    for publico, patron in _PUBLICOS.items():
        if re.search(patron, texto):
            return publico
    return None

def _clasificar_publico(row):
    # Si el NOMBRE de la carrera ya dice "equipos" ("... POR EQUIPOS",
    # "CAMPIONATO ... EQUIPOS"), es una prueba por equipos aunque la
    # modalidad diga otra cosa (p. ej. "ABSOLUTA" o "FEMENINA") — este
    # indicador manda por encima de cualquier otro match.
    nombre = row["nombre_carrera"]
    texto_nombre = "" if pd.isna(nombre) else nombre.lower()
    if re.search(r"equipos?\b", texto_nombre):
        return "Equipos"

    modalidad = row["modalidad"]
    texto_modalidad = "" if pd.isna(modalidad) else modalidad.lower().strip()

    publico = _clasificar_publico_texto(texto_modalidad) if texto_modalidad else None
    if publico:
        return publico

    publico = _clasificar_publico_texto(texto_nombre)
    if publico:
        return publico

    if texto_modalidad and re.search(_OTROS_PATRON, texto_modalidad):
        return "Otros"
    return "Absoluta/General"

curses_limpio["publico"] = curses_limpio.apply(_clasificar_publico, axis=1)

print(curses_limpio["publico"].value_counts())
print()
print("Valores de 'modalidad' que han caído por defecto en 'Absoluta/General':")
print(curses_limpio.loc[curses_limpio["publico"] == "Absoluta/General", "modalidad"].value_counts().head(30))
print()
print("Valores de 'modalidad' que han caído en 'Otros':")
print(curses_limpio.loc[curses_limpio["publico"] == "Otros", "modalidad"].value_counts())

publico
Absoluta/General     4030
Infantil             1733
Equipos               269
Elite                 118
Mayores/Veteranos      87
Otros                  43
Name: count, dtype: int64

Valores de 'modalidad' que han caído por defecto en 'Absoluta/General':
modalidad
ABSOLUTA                    433
                            301
PROBA ABSOLUTA              166
ADULTOS                     136
TRAIL                        85
POPULAR                      66
CARREIRA ABSOLUTA            61
PRUEBA ABSOLUTA              59
CARREIRA                     53
MEDIO MARATON                42
FEMENINA                     40
RECORRIDO                    38
Carrera                      36
MASCULINA                    35
Femenina                     34
PRUEBA                       34
CATEGORIA B                  32
CATEGORIA A                  31
RECORRIDO B                  30
INDIVIDUAL                   29
CATEGORIA C                  26
Adultos                      25
CONTRA O CANCRO        

### Categorías finales

**`tipo_modalidad`** (disciplina): road running, trail running, Ciclismo y btt, Multidisciplina, marcha, Otros (residual).

**`publico`** (edad/tipo de participante): Infantil (fusiona Infantil+Cadete+Juvenil), Mayores/Veteranos, Elite, Equipos, Absoluta/General (por defecto), Otros.

### Ejemplos de lo que queda en "Otros"

Muestra de filas completas (no solo el texto de `modalidad`) para revisar de un vistazo qué tipo de casos se quedan sin clasificar en cada columna.

In [8]:
pd.set_option("display.max_colwidth", None)

cols = ["nombre_carrera", "modalidad", "distancia", "tipo_modalidad", "publico"]

print(f"tipo_modalidad == 'Otros': {(curses_limpio['tipo_modalidad'] == 'Otros').sum()} filas — muestra:")
display(curses_limpio.loc[curses_limpio["tipo_modalidad"] == "Otros", cols].sample(15, random_state=0))

print()
print(f"publico == 'Otros': {(curses_limpio['publico'] == 'Otros').sum()} filas — muestra:")
display(curses_limpio.loc[curses_limpio["publico"] == "Otros", cols].sample(15, random_state=0))

tipo_modalidad == 'Otros': 883 filas — muestra:


,nombre_carrera,modalidad,distancia,tipo_modalidad,publico
3284,TRAVESIA LA HEROICA GRAN PREMIO DEPUTACION DE LUGO. 7ª et III Copa de España de Aguas Abiertas,TRAVESIA DE SIN NEOPRENO,5.0,Otros,Absoluta/General
4199,VII TRAVESÍA A NADO BENÉFICA DE LOS INOCENTES,TRAVESÍA A NADO,NaN,Otros,Absoluta/General
4705,XACOBEO WILD RACE 2022,WOLF EQUIPOS FEMENINOS,NaN,Otros,Equipos
2442,IV ESCALADA MONTE DE A CARBALLEIRA,EQUIPOS,NaN,Otros,Equipos
3176,SAMURAI XTREME RACE AS PONTES 2022,Liga Open Grupos de Edad,NaN,Otros,Absoluta/General
1016,ETAPA FINAL CIRCUITO GALLEGO AA ARZUA ULLOA OPEN 2025- CAMPEONATO GALLEGO CONTRARRELOJ EQUIPOS,MASTER,NaN,Otros,Equipos
3382,Upstream Lago de Sanabria 2015,Equipos,6.5,Otros,Equipos
377,81 TRAVESIA A NADO Á ENSEADA DE SAN AMARO,Travesía,1.2,Otros,Absoluta/General
2423,IV EDICION 24 HORAS DE VIGO,EQUIPOS MIXTOS,NaN,Otros,Equipos
2728,IX Travesia a Nado de Larga Distancia A Coruña 10000 y I Coruña 5000 - 2016,TRAVESIA,10.0,Otros,Absoluta/General



publico == 'Otros': 43 filas — muestra:


,nombre_carrera,modalidad,distancia,tipo_modalidad,publico
5668,XXI Medio Maratón - II Maratón Gran Bahía VIG-BAY,"MARATON TRICICLO, HANDBIKE Y JOELETTE",42.195,trail running,Otros
5829,XXIV MEMORIAL ADOLFO ROS - VOLTA A RIA 2009,CADEIRA DE RODAS,21.097,road running,Otros
4151,VII MEDIA MARATON CIUDAD DE LEÓN,MEDIO MARATÓN EN SILLA DE RUEDAS,21.097,road running,Otros
489,CAMPEONATO DE ESPAÑA CICLISMO ADAPTADO RUTA,HANDBIKE Y TRICICLES,20.000,Ciclismo y btt,Otros
1361,I MARCHA-CARRERA POR LA IGUALDAD CIUDAD DE PONTEVEDRA,Marcha para personas con discapacidad,NaN,marcha,Otros
4098,VII Carreira Pedestre Concello de Vimianzo,CADEIRA DE RODAS,NaN,road running,Otros
5177,XIX MEDIO MARATÓN GRAN BAHÍA VIG-BAY edp,"TRICICLO, HANDBIKE Y JOELETTE",21.097,trail running,Otros
1574,II CARREIRA SOLIDARIA ABANCA NOCTURNA DAS FESTAS DE LALIN,HANDBIKE,NaN,Ciclismo y btt,Otros
5834,XXIV Medio Maratón Gran Bahía VIG-BAY,"MEDIO MARATON TRICICLO, HANDBIKE Y JOELETTE",21.097,trail running,Otros
5669,XXI Medio Maratón - II Maratón Gran Bahía VIG-BAY,"MEDIO MARATON TRICICLO, HANDBIKE Y JOELETTE",21.097,trail running,Otros


### Esquema común entre las 10 fuentes

Para poder comparar o concatenar directamente las tablas de buscametas, xipgroc, carreirasgalegas, ccnorte, cronofinisher, mychip, sportmaniacs, cursescat, iter5 y cruzandolameta, las 12 columnas que comparten todas van con el mismo nombre y en el mismo orden: `fuente`, `nombre_carrera`, `fecha`, `dia_semana`, `distancia`, `tipo_modalidad`, `publico`, `finisher_d`, `finisher_h`, `municipio`, `comarca`, `provincia` (antes `ok_d`/`ok_h` aquí, y sin ubicación). `fuente` es una constante ("ccnorte") para identificar el origen al concatenar las 10 tablas. `municipio`/`comarca`/`provincia` se obtienen reutilizando la geocodificación que ya hizo `Scraper_ccnorte.ipynb` (Fase 3, `ccnorte_ubicaciones.csv`) — ver más abajo. Lo que es propio solo de ccnorte (`finisher_desconocido`, `categoria_genero`, `modalidad`) va al final.

### Ubicación — reutilizando la geocodificación del scraper

`ccnorte_ubicaciones.csv` ya existe (lo genera `Scraper_ccnorte.ipynb`, Fase 3): para cada `nombre_carrera` intentó extraer el lugar del propio nombre y geocodificarlo con Nominatim, guardando `candidato_lugar`/`lat`/`lon`, pero solo como una dirección de texto suelta (`ubicacion`), sin `municipio`/`comarca`/`provincia` ya separados.

Aquí lo aprovechamos en dos pasos, con checkpoint propio en `ccnorte_municipios.csv`:
1. Para las carreras que ya tienen `lat`/`lon` (algo más de la mitad), hacemos **reverse geocoding** de esas coordenadas — no hace falta gastar otra petición de búsqueda por texto, y con `addressdetails=True` nos da `municipio`/`comarca`/`provincia` ya estructurados.
2. Para las que fallaron en el scraper (sin `lat`/`lon`), reintentamos hacia delante con la misma heurística de encoger el candidato que usamos en xipgroc (nunca por debajo de 2 palabras, para no acabar geocodificando una palabra suelta sin relación con la carrera).

`comarca` ("county" en OpenStreetMap) suele salir vacía fuera de Galicia (aquí hay carreras de varias comunidades, no solo Galicia como en carreirasgalegas) — no es un fallo, es que esa etiqueta administrativa no está bien mapeada en OSM en esas zonas.

Es lento (~2500 carreras únicas, 1 petición/segundo) pero tiene checkpoint: se puede interrumpir y continuar.

In [9]:
import csv
import time


def _geocode_con_reintentos(geolocator, candidato, pausa_segundos):
    from geopy.exc import GeopyError

    tokens = candidato.split()
    max_start = max(0, len(tokens) - 2) if len(tokens) > 1 else 0

    for start in range(max_start + 1):
        query = " ".join(tokens[start:])
        if not query:
            break
        try:
            loc = geolocator.geocode(
                f"{query}, España", exactly_one=True, country_codes="es",
                addressdetails=True, timeout=10,
            )
        except GeopyError:
            loc = None
        if loc:
            return loc
        if start < max_start:
            time.sleep(pausa_segundos)
    return None


def _direccion_a_municipio_comarca_provincia(addr):
    municipio = addr.get("city") or addr.get("town") or addr.get("village") or addr.get("municipality")
    comarca = addr.get("county")
    provincia = addr.get("province") or addr.get("state")
    return municipio, comarca, provincia


def geocodificar_municipios_ccnorte(nombres_carrera, out_dir, pausa_segundos: float = 1.1):
    from geopy.geocoders import Nominatim
    from geopy.exc import GeopyError

    out_path = Path(out_dir)
    csv_previo = out_path / "ccnorte_ubicaciones.csv"  # generado por el scraper
    csv_ubic = out_path / "ccnorte_municipios.csv"  # checkpoint propio de este notebook

    previo = pd.read_csv(csv_previo, dtype=str) if csv_previo.exists() else pd.DataFrame(
        columns=["nombre_carrera", "candidato_lugar", "ubicacion", "lat", "lon"]
    )
    previo_por_nombre = previo.set_index("nombre_carrera").to_dict("index")

    cache = {}
    if csv_ubic.exists():
        prev = pd.read_csv(csv_ubic, dtype=str)
        cache = {row["nombre_carrera"]: row.to_dict() for _, row in prev.iterrows()}
        print(f"Checkpoint: {len(cache)} carreras ya resueltas")

    geolocator = Nominatim(user_agent="ccnorte_municipios_claudia")

    nombres_unicos = list(dict.fromkeys(n for n in nombres_carrera if isinstance(n, str)))
    pendientes = [n for n in nombres_unicos if n not in cache]
    print(f"Carreras a resolver: {len(pendientes)} (de {len(nombres_unicos)} únicas)")

    campos = ["nombre_carrera", "municipio", "comarca", "provincia"]
    write_header = not csv_ubic.exists()
    with open(csv_ubic, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=campos)
        if write_header:
            writer.writeheader()

        for i, nombre in enumerate(pendientes, 1):
            fila = {"nombre_carrera": nombre, "municipio": None, "comarca": None, "provincia": None}
            info = previo_por_nombre.get(nombre)
            try:
                if info and pd.notna(info.get("lat")) and pd.notna(info.get("lon")):
                    loc = geolocator.reverse(
                        (float(info["lat"]), float(info["lon"])),
                        exactly_one=True, addressdetails=True, timeout=10,
                    )
                else:
                    candidato = (
                        info["candidato_lugar"]
                        if info and pd.notna(info.get("candidato_lugar"))
                        else nombre
                    )
                    loc = _geocode_con_reintentos(geolocator, candidato, pausa_segundos)

                if loc:
                    municipio, comarca, provincia = _direccion_a_municipio_comarca_provincia(
                        loc.raw.get("address", {})
                    )
                    fila["municipio"] = municipio
                    fila["comarca"] = comarca
                    fila["provincia"] = provincia
            except GeopyError as e:
                print(f"  [{nombre}] error de geocodificación: {e}")
            except Exception as e:
                print(f"  [{nombre}] ERROR inesperado: {e}")

            writer.writerow(fila)
            f.flush()
            cache[nombre] = fila
            time.sleep(pausa_segundos)  # respeta el límite de Nominatim (1 req/s)

            if i % 100 == 0 or i == len(pendientes):
                print(f"  Progreso: {i}/{len(pendientes)}")

    print(f"CSV de municipios: {csv_ubic.resolve()}")
    df_ubic = pd.DataFrame(cache.values())
    print(f"Con municipio: {df_ubic['municipio'].notna().sum()} de {len(df_ubic)}")
    return df_ubic


OUT_DIR = Path("../../data/raw/ccnorte")
df_municipios = geocodificar_municipios_ccnorte(curses_limpio["nombre_carrera"], out_dir=OUT_DIR)

curses_limpio = curses_limpio.merge(df_municipios, on="nombre_carrera", how="left")

print("Filas con municipio:", curses_limpio["municipio"].notna().sum(), "de", len(curses_limpio))
curses_limpio.sample(15)

Checkpoint: 2479 carreras ya resueltas
Carreras a resolver: 0 (de 2479 únicas)
CSV de municipios: C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\ccnorte_data\ccnorte_municipios.csv
Con municipio: 1791 de 2479
Filas con municipio: 4540 de 6280


,nombre_carrera,fecha,modalidad,categoria_genero,finisher_d,finisher_h,finisher_desconocido,distancia,tipo_modalidad,publico,municipio,comarca,provincia
4845,XI TRAIL CONCELLO DE ARTEIXO,2024-10-27,CURTA,mixto,44,126,0,NaN,trail running,Absoluta/General,Arteixo,A Coruña,Galicia
5707,XXII CARRERA POPULAR CONCELLO DE ARTEIXO,2019-05-01,HANDBIKE / SILLA DE ATLETISMO,mixto,0,4,0,NaN,Ciclismo y btt,Absoluta/General,Arteixo,A Coruña,Galicia
3589,V Edición terremoto trail,2016-10-23,MEDIA,mixto,4,45,0,21.0,trail running,Absoluta/General,Santiago de Compostela,Santiago,Galicia
2544,IX CAMPEONATO GALLEGO DE AGUAS ABIERTAS,2025-06-01,MASTER,mixto,44,66,0,2.5,Otros,Mayores/Veteranos,NaN,NaN,NaN
449,AGUAS ABIERTAS (AF) CASTRELO DE MIÑO,2024-09-21,FEMENINO,femenino,63,0,0,NaN,Otros,Absoluta/General,A Carreira,Betanzos,Galicia
4108,VII Carreira Popular de Taboada “Entre pazos e carballos”,2018-09-22,CATEGORÍA C,mixto,9,24,0,NaN,road running,Absoluta/General,Betanzos,Betanzos,Galicia
1991,III CARRERA POPULAR MATOGRANDE,2018-01-14,SUB16 y SUB18,mixto,90,54,0,NaN,road running,Infantil,A Coruña,A Coruña,Galicia
4014,"VII CARREIRA POPULAR DO SAN SALVADOR, Baños de Molgas MENORES",2014-08-02,MENORES,mixto,0,0,0,2.3,road running,Infantil,NaN,NaN,NaN
5633,XXI CARREIRA POPULAR CONCELLO DE CARTELLE,2017-08-19,ALEVIN,mixto,13,12,0,NaN,road running,Infantil,Outomuro,Terra da Celanova,Ourense
1599,II CARRERA POPULAR 10K DE TEIS,2020-03-01,,mixto,91,337,0,10.0,road running,Absoluta/General,Vigo,Vigo,Galicia


In [10]:
# "distancia" se quedó en NaN (no 0) cuando no se pudo extraer — la
# rellenamos a 0 para que use el mismo valor centinela "sin distancia"
# que el resto de fuentes (si no, mezclar NaN y 0 como "desconocido" entre
# fuentes rompe cualquier filtro/agregado que se haga sobre la tabla
# unida de las 10 fuentes).
curses_limpio["distancia"] = curses_limpio["distancia"].fillna(0)

# Añadimos "fuente" (constante, para identificar el origen al concatenar
# con las otras 9 tablas) y "dia_semana" (derivado de "fecha"), y
# reordenamos las columnas para que el esquema común (fuente,
# nombre_carrera, fecha, dia_semana, distancia, tipo_modalidad, publico,
# finisher_d, finisher_h, municipio, comarca, provincia) quede igual en
# las 10 fuentes, dejando lo propio de ccnorte (finisher_desconocido,
# categoria_genero, modalidad) al final.
curses_limpio["fuente"] = "ccnorte"

_DIAS_SEMANA = ["Lunes", "Martes", "Miércoles", "Jueves", "Viernes", "Sábado", "Domingo"]
curses_limpio["dia_semana"] = curses_limpio["fecha"].dt.dayofweek.map(dict(enumerate(_DIAS_SEMANA)))

curses_limpio = curses_limpio[
    ["fuente", "nombre_carrera", "fecha", "dia_semana", "distancia", "tipo_modalidad", "publico",
     "finisher_d", "finisher_h", "municipio", "comarca", "provincia",
     "finisher_desconocido", "categoria_genero", "modalidad"]
]
curses_limpio.columns.tolist()

['fuente',
 'nombre_carrera',
 'fecha',
 'dia_semana',
 'distancia',
 'tipo_modalidad',
 'publico',
 'finisher_d',
 'finisher_h',
 'municipio',
 'comarca',
 'provincia',
 'finisher_desconocido',
 'categoria_genero',
 'modalidad']

In [11]:
# Guardamos la tabla ya limpia y clasificada para poder descargarla.
SALIDA = Path("../../data/processed/ccnorte/DF_CCNORTE_LIMPIO.csv")
curses_limpio.to_csv(SALIDA, index=False, encoding="utf-8-sig")
print(f"Guardado en {SALIDA}")

Guardado en C:\Users\Clàudia Rafart\OneDrive - Prenomics\Escritorio\Altres\ccnorte_data\DF_CCNORTE_LIMPIO.csv
